# POC 5: The Change Communication Drafter

**Pain point:** Any process or policy change needs a communication to affected teams. Written without grounding, it's vague enough to trigger a flood of follow-up questions the email should have answered. Written with wrong information (generic model output), it causes more confusion than the original change.

**What this notebook shows:** The grounded model is asked to *prevent questions*, not answer them. It's given the change document, the FAQ, and the list of who's affected — and asked to preemptively answer what each group will ask. The ungrounded model produces the 'we're making some changes' email everyone ignores.

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
!pip install groq -q

In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

client = Groq(api_key=GROQ_API_KEY)
MODEL = 'openai/gpt-oss-20b'

def call_llm(system_prompt, user_message, temperature=0.4, max_tokens=900):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'Model ready: {MODEL}')

In [ ]:
# --- Synthetic grounding material ---

CHANGE_DOCUMENT = """
POLICY CHANGE: Expense Management System Migration
Effective date: 1 November 2025
Owner: Finance Operations (contact: expenses@company.com)

What is changing:
  - All expense claims over £50 must be pre-approved via the new Concur system before the expense is incurred.
  - Receipts under £50 can continue to be submitted in the monthly expense report (no change).
  - Mileage claims now require a GPS log exported from the Concur mobile app.
    Manual mileage logs will not be accepted for claims submitted after 1 November.
  - The existing expenses spreadsheet will be decommissioned on 31 October.

What is NOT changing:
  - Per-diem rates remain unchanged.
  - Approval chains remain the same — your line manager approves, Finance countersigns.
  - Reimbursement timelines remain the same (next payroll cycle).
  - Expenses incurred before 1 November should still be submitted via the old process.
"""

FAQ = """
FINANCE OPS FAQ — Expense System Migration

Q: Do I need to retroactively move old expenses to Concur?
A: No. Only claims for expenses incurred on or after 1 November 2025 use Concur.

Q: What if I don't have a smartphone for the GPS mileage log?
A: Contact expenses@company.com before 25 October to request a manual mileage form exemption.
   Exemptions are granted case by case.

Q: Is the £50 threshold per receipt or per trip?
A: Per receipt. Multiple receipts under £50 from the same trip do not need pre-approval individually.

Q: What if I need to incur an expense urgently and can't get pre-approval in time?
A: Contact your line manager who can grant a post-hoc approval within 48 hours. Document the
   reason in the Concur notes field.

Q: When will Concur training be available?
A: Two 30-minute live sessions on 22 and 24 October. Recording will be on the intranet.
   Mandatory for field sales and consultants. Optional for others.
"""

AFFECTED_GROUPS = """
AFFECTED GROUPS AND SPECIFIC IMPACTS:

Field Sales (32 people):
  - Daily mileage claims: highest impact group. Every trip > £50 now needs pre-approval.
  - GPS log requirement affects them most — they drive frequently.
  - Training is mandatory for this group.

Consultants / Client Services (18 people):
  - Frequent client entertainment and travel. Many claims are > £50.
  - Some do not have company smartphones — manual exemption may be needed.
  - Training is mandatory for this group.

Office-based staff (all others):
  - Low-frequency claimants. Most claims are under £50.
  - Minimal impact. Training optional.
"""

FULL_CONTEXT = f"""
{CHANGE_DOCUMENT}

{FAQ}

{AFFECTED_GROUPS}
"""

print('Grounding data loaded.')

In [ ]:
# --- UNGROUNDED call ---

ungrounded_system = "You are an internal communications assistant."

ungrounded_query = """
We are changing our expense management system to Concur from 1 November.
Draft an all-staff email announcing the change.
"""

print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, ungrounded_query))

In [ ]:
# --- GROUNDED call ---

grounded_system = """
You are an internal communications specialist. Your goal is to draft a communication
that prevents follow-up questions — not just announces the change.

Rules:
- State clearly what IS changing and what is NOT changing
- Acknowledge the specific groups most affected and what changes for them
- Pre-answer the top 3 questions people will actually ask (based on the FAQ)
- Include the action each group needs to take before the go-live date
- One clear contact for questions — do not say 'reach out to your manager'
- Tone: direct and helpful, not corporate. People read this on a phone between meetings.
- Length: under 300 words. No generic opener ('I'm pleased to announce...')
"""

grounded_query = f"""
Draft an all-staff email announcing the expense system migration to Concur from 1 November.

{FULL_CONTEXT}
"""

print('=== GROUNDED OUTPUT ===')
print(call_llm(grounded_system, grounded_query))

In [ ]:
# --- BONUS: Targeted email for field sales specifically ---
# Same grounding, different output instruction.

targeted_system = """
You are an internal communications specialist.
Write a short, direct message for a specific audience group.
They already know the system is changing — focus only on what changes for THEM.
Under 150 words. Bullet format. Include the two actions they must take.
"""

targeted_query = f"""
Write the targeted message for Field Sales only.
They are the highest-impact group for this change.

{FULL_CONTEXT}
"""

print('=== TARGETED: Field Sales Message ===')
print(call_llm(targeted_system, targeted_query, max_tokens=300))

## What just happened

The **ungrounded** output is the email everyone skims and ignores — 'We are excited to announce the migration to Concur... please visit the intranet for more details.' It doesn't tell field sales they need to attend mandatory training. It doesn't tell the consultant without a smartphone they need to request an exemption by 25 October. It doesn't mention the GPS mileage log at all.

The **grounded** output answers the questions before they're asked — which is the actual job of a change communication.

The **bonus cell** demonstrates a second grounding pattern: same context, different output constraint, different audience. You don't re-load the data. You re-use the grounding and change the instruction. This is token-efficient — the expensive part (the context) is written once; the cheap part (the instruction) controls what you get back.

**The real grounding insight here:** The model wasn't asked 'what does this document say?' It was asked 'what will people ask about this, and answer those questions preemptively.' That's a different prompt pattern — and it's what makes this useful rather than just technically correct.